In [ ]:
# -*- coding: utf-8 -*-
"""ConvNeXt_Segmentation.ipynb"""
from google.colab import drive
drive.flush_and_unmount()
print("Drive unmounted")

Drive unmounted


In [1]:
# =============================================================================
# ConvUNeXt — 3-Class Segmentation | Colab | T4 GPU | PyTorch
#
# Folder layout (relative to os.getcwd()):
#   class_1/          ← images for class 1  (RGB or RGBA)
#   class_1_mask/     ← binary masks  (white px = object, black = background)
#   class_2/          ← images for class 2  (RGB or RGBA)
#   class_2_mask/     ← binary masks  (white px = object, black = background)
#   class_3/          ← negative / background-only tiles  (no masks needed)
#
# Label map in the segmentation output:
#   0 → background  (black pixels in masks + all of class_3)
#   1 → class 1 object pixels  (white pixels in class_1_mask)
#   2 → class 2 object pixels  (white pixels in class_2_mask)
#
# Notes:
#   • If images or masks are 32-bit RGBA, the alpha channel is stripped
#     automatically before any processing.
#   • Mask binarisation threshold = 127  (pixel > 127 → foreground).
#   • Early stopping monitors val_loss with configurable patience.
# =============================================================================

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive') # ← You will be asked to authorize

Mounted at /content/drive


In [ ]:
# Set your working folder inside Drive
import os
os.chdir('/content/drive/MyDrive/trees') # ← CHANGE THIS TO YOUR FOLDER

In [3]:
# Install Dependencies
!pip install -q timm albumentations

In [5]:
# Imports
import os
import cv2
import time
import timm
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset, random_split
from torch.cuda.amp import GradScaler, autocast

import albumentations as A
from albumentations.pytorch import ToTensorV2

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device : {DEVICE}")
print(f"✅ GPU    : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

✅ Device : cuda
✅ GPU    : Tesla T4


In [8]:
# Config
class CFG:
    # ── Folder names (relative to cwd) ─────────────────────────────────────
    CLASS1_IMG_DIR   = "class_1"
    CLASS1_MASK_DIR  = "class_1_mask"
    CLASS2_IMG_DIR   = "class_2"
    CLASS2_MASK_DIR  = "class_2_mask"
    CLASS3_IMG_DIR   = "class_3"         # no mask folder — all background

    # ── Output folders ──────────────────────────────────────────────────────
    CKPT_DIR         = "checkpoints"
    LOG_DIR          = "logs"

    # ── Model ──────────────────────────────────────────────────────────────
    BACKBONE         = "convnext_small"   # convnext_tiny | small | base | large
    PRETRAINED       = True
    NUM_CLASSES      = 3                 # 0=background, 1=class1, 2=class2

    # ── Training ───────────────────────────────────────────────────────────
    IMG_SIZE         = 192
    BATCH_SIZE       = 16                 # comfortable on T4 @ 512 px
    NUM_EPOCHS       = 800               # upper bound — early stopping may halt sooner
    LR               = 3e-4
    WEIGHT_DECAY     = 1e-4
    AMP              = True
    NUM_WORKERS      = 2

    # ── Early Stopping ─────────────────────────────────────────────────────
    ES_PATIENCE      = 50               # stop after N epochs with no improvement
    ES_MIN_DELTA     = 1e-4             # minimum change in val_loss to count as improvement
    ES_RESTORE_BEST  = True             # reload best weights before returning

    # ── Loss ───────────────────────────────────────────────────────────────
    # Class weights for CrossEntropy.  [bg, class1, class2]
    # Increase class1/class2 weights if objects are small relative to the tile.
    CE_CLASS_WEIGHTS = [0.3, 1.0, 1.0]
    CE_WEIGHT        = 0.5
    DICE_WEIGHT      = 0.5

    # ── Scheduler ───────────────────────────────────────────────────────────
    LR_MIN           = 1e-6

    # ── Val split ───────────────────────────────────────────────────────────
    VAL_SPLIT        = 0.15

    SAVE_BEST        = True              # save best val-loss checkpoint


os.makedirs(CFG.CKPT_DIR, exist_ok=True)
os.makedirs(CFG.LOG_DIR,  exist_ok=True)


# RGBA-safe Image & Mask Loaders
def load_image_rgb(path: str) -> np.ndarray:
    """
    Load any image as a 3-channel uint8 RGB array.
    Handles both RGB (24-bit) and RGBA (32-bit) inputs:
      • RGBA → alpha channel is discarded, only RGB is returned.
      • Greyscale → converted to 3-channel RGB.
    """
    img = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)

    if img is None:
        raise FileNotFoundError(f"Could not read image: {path}")

    if img.ndim == 2:
        # Greyscale → RGB
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    elif img.shape[2] == 4:
        # RGBA (32-bit) → drop alpha, convert BGR→RGB
        img = cv2.cvtColor(img, cv2.COLOR_BGRA2RGB)
    else:
        # Standard BGR (24-bit) → RGB
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    return img.astype(np.uint8)


def load_mask_binary(path: str, fg_label: int,
                     threshold: int = 127) -> np.ndarray:
    """
    Load a binary mask and return an int64 label map.

    Mask convention (as specified):
      • White pixels (value > threshold) → foreground → assigned `fg_label`
      • Black pixels (value ≤ threshold) → background → label 0

    Handles both greyscale masks and RGBA masks:
      • Greyscale (8-bit)  → used directly.
      • RGBA (32-bit mask) → only the R channel is used for thresholding;
        the alpha channel is ignored.
      • RGB (24-bit)       → converted to greyscale before thresholding.
    """
    mask = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)

    if mask is None:
        raise FileNotFoundError(f"Could not read mask: {path}")

    if mask.ndim == 2:
        # Already greyscale — use as-is
        grey = mask
    elif mask.shape[2] == 4:
        # RGBA mask — discard alpha, take mean of RGB as luminance
        grey = cv2.cvtColor(mask[:, :, :3], cv2.COLOR_BGR2GRAY)
    else:
        # BGR mask
        grey = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)

    # White  (> threshold) → fg_label  |  Black (≤ threshold) → 0
    label_map = np.where(grey > threshold, fg_label, 0).astype(np.int64)
    return label_map


# Augmentation Pipelines
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_transform = A.Compose([
    A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1,
                       rotate_limit=20, p=0.5),
    A.OneOf([
        A.GaussianBlur(blur_limit=3),
        A.MotionBlur(blur_limit=3),
    ], p=0.2),
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2),
        A.HueSaturationValue(hue_shift_limit=10,
                              sat_shift_limit=20, val_shift_limit=10),
    ], p=0.4),
    A.CoarseDropout(max_holes=8, max_height=32, max_width=32,
                    fill_value=0, mask_fill_value=0, p=0.2),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

SUPPORTED_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}


# Datasets
class MaskedClassDataset(Dataset):
    """
    For class_1 and class_2:
      - Loads image via load_image_rgb()  → strips alpha if RGBA.
      - Loads mask  via load_mask_binary() → strips alpha, applies threshold.
          white pixels (> 127) → fg_label
          black pixels (≤ 127) → 0  (background)
    """

    def __init__(self, img_dir: str, mask_dir: str,
                 class_label: int, transform=None):
        self.class_label = class_label
        self.transform   = transform

        self.img_paths   = sorted([
            p for p in Path(img_dir).iterdir()
            if p.suffix.lower() in SUPPORTED_EXTS
        ])
        self.mask_paths  = sorted([
            p for p in Path(mask_dir).iterdir()
            if p.suffix.lower() in SUPPORTED_EXTS
        ])

        assert len(self.img_paths) == len(self.mask_paths), (
            f"[Class {class_label}] Image/mask count mismatch: "
            f"{len(self.img_paths)} vs {len(self.mask_paths)}"
        )
        print(f"  📂 Class {class_label} : {len(self.img_paths)} pairs  "
              f"('{img_dir}' ↔ '{mask_dir}')")

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        image = load_image_rgb(self.img_paths[idx])        # (H, W, 3) uint8
        mask  = load_mask_binary(self.mask_paths[idx],     # (H, W)    int64
                                 fg_label=self.class_label)

        if self.transform:
            aug   = self.transform(image=image, mask=mask)
            image = aug["image"]        # float32 tensor (3, H, W)
            mask  = aug["mask"].long()  # int64  tensor (H, W)

        return image.float(), mask


class BackgroundDataset(Dataset):
    """
    For class_3 (negative / background-only tiles):
      - Loads images only via load_image_rgb() → strips alpha if RGBA.
      - Returns an all-zero mask (label = 0 everywhere).
    """

    def __init__(self, img_dir: str, transform=None):
        self.transform = transform
        self.img_paths = sorted([
            p for p in Path(img_dir).iterdir()
            if p.suffix.lower() in SUPPORTED_EXTS
        ])
        print(f"  📂 Class 3 (bg) : {len(self.img_paths)} images  "
              f"('{img_dir}', no masks — all-zero labels)")

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        image = load_image_rgb(self.img_paths[idx])        # (H, W, 3) uint8
        H, W  = image.shape[:2]
        mask  = np.zeros((H, W), dtype=np.int64)           # all background

        if self.transform:
            aug   = self.transform(image=image, mask=mask)
            image = aug["image"]
            mask  = aug["mask"].long()

        return image.float(), mask


def build_loaders():
    print("=" * 60)
    print("  Building datasets …")
    print("=" * 60)

    ds1  = MaskedClassDataset(CFG.CLASS1_IMG_DIR, CFG.CLASS1_MASK_DIR,
                               class_label=1)
    ds2  = MaskedClassDataset(CFG.CLASS2_IMG_DIR, CFG.CLASS2_MASK_DIR,
                               class_label=2)
    ds3  = BackgroundDataset(CFG.CLASS3_IMG_DIR)
    full = ConcatDataset([ds1, ds2, ds3])

    n_val   = max(1, int(len(full) * CFG.VAL_SPLIT))
    n_train = len(full) - n_val

    train_ds, val_ds = random_split(
        full, [n_train, n_val],
        generator=torch.Generator().manual_seed(SEED)
    )

    # Patch transforms after split — val images never see augmentation
    def _set_transforms(subset, tfm):
        for ds in subset.dataset.datasets:
            ds.transform = tfm

    _set_transforms(train_ds, train_transform)
    _set_transforms(val_ds,   val_transform)

    train_loader = DataLoader(
        train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True,
        num_workers=CFG.NUM_WORKERS, pin_memory=True, drop_last=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=CFG.BATCH_SIZE, shuffle=False,
        num_workers=CFG.NUM_WORKERS, pin_memory=True
    )

    print(f"\n  🔀 Total : {len(full)} | Train : {n_train} | Val : {n_val}")
    print(f"      Class 1 : {len(ds1)} | Class 2 : {len(ds2)} "
          f"| Class 3 (bg) : {len(ds3)}")
    return train_loader, val_loader


# ConvUNeXt Architecture
class ConvBnAct(nn.Module):
    def __init__(self, in_ch, out_ch, kernel=3, act=True):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel,
                      padding=kernel // 2, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.GELU() if act else nn.Identity(),
        )
    def forward(self, x):
        return self.block(x)


class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up   = nn.Upsample(scale_factor=2, mode="bilinear",
                                 align_corners=False)
        self.conv = nn.Sequential(
            ConvBnAct(in_ch + skip_ch, out_ch),
            ConvBnAct(out_ch, out_ch),
        )

    def forward(self, x, skip=None):
        x = self.up(x)
        if skip is not None:
            if x.shape[-2:] != skip.shape[-2:]:
                x = F.interpolate(x, size=skip.shape[-2:],
                                  mode="bilinear", align_corners=False)
            x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class ConvUNeXt(nn.Module):
    """
    ConvNeXt encoder (ImageNet-pretrained via timm) + U-Net decoder.
    Output: (B, NUM_CLASSES, H, W) raw logits for CrossEntropyLoss.

    Encoder feature map strides (convnext_tiny example):
      s0 : H/4  — C=96
      s1 : H/8  — C=192
      s2 : H/16 — C=384
      s3 : H/32 — C=768
    """

    """ENCODER_CHANNELS = {
        "convnext_tiny" : [96,  192,  384,  768],
        "convnext_small": [96,  192,  384,  768],
        "convnext_base" : [128, 256,  512, 1024],
        "convnext_large": [192, 384,  768, 1536],
    }
    DECODER_CHANNELS = [256, 128, 64, 32]"""

    def __init__(self, backbone="convnext_tiny",
                 pretrained=True, num_classes=3):
        super().__init__()
        """enc_chs = self.ENCODER_CHANNELS[backbone]
        dec_chs = self.DECODER_CHANNELS

        self.encoder = timm.create_model(
            backbone, pretrained=pretrained,
            features_only=True, out_indices=(0, 1, 2, 3),
        )"""
        # Build encoder first — timm exposes the actual channel widths
        # of whatever backbone was constructed via feature_info.channels()
        # No manual lookup table needed; works for any backbone timm supports
        self.encoder = timm.create_model(
            backbone, pretrained=pretrained,
            features_only=True, out_indices=(0, 1, 2, 3),
        )
        enc_chs = self.encoder.feature_info.channels()
        # enc_chs is now e.g. [96, 192, 384, 768] for convnext_tiny,
        # read directly from the model — no dict required

        # Decoder channels derived from the deepest encoder channel:
        # start at half the bottleneck width, halve at each stage
        # e.g. convnext_tiny bottleneck=768 → dec_chs=[384,192,96,48]
        # e.g. convnext_base bottleneck=1024 → dec_chs=[512,256,128,64]
        # This keeps the decoder proportional to the encoder automatically
        bottleneck = enc_chs[-1]
        dec_chs = [bottleneck // 2, bottleneck // 4,
                   bottleneck // 8, bottleneck // 16]
        print(f"  🧠 Encoder : {backbone}  (pretrained={pretrained})")
        print(f"  🔢 Encoder channels : {enc_chs}")
        print(f"  🔢 Decoder channels : {dec_chs}")

        self.dec4 = DecoderBlock(enc_chs[3], enc_chs[2], dec_chs[0])
        self.dec3 = DecoderBlock(dec_chs[0], enc_chs[1], dec_chs[1])
        self.dec2 = DecoderBlock(dec_chs[1], enc_chs[0], dec_chs[2])
        self.dec1 = DecoderBlock(dec_chs[2], 0,          dec_chs[3])

        self.head = nn.Sequential(
            ConvBnAct(dec_chs[3], dec_chs[3]),
            nn.Conv2d(dec_chs[3], num_classes, kernel_size=1),
        )

    def forward(self, x):
        H, W           = x.shape[-2:]
        s0, s1, s2, s3 = self.encoder(x)
        d4 = self.dec4(s3, s2)
        d3 = self.dec3(d4, s1)
        d2 = self.dec2(d3, s0)
        d1 = self.dec1(d2)
        return F.interpolate(self.head(d1), size=(H, W),
                             mode="bilinear", align_corners=False)


def build_model():
    model = ConvUNeXt(
        backbone    = CFG.BACKBONE,
        pretrained  = CFG.PRETRAINED,
        num_classes = CFG.NUM_CLASSES,
    ).to(DEVICE)
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  📐 Trainable params : {n / 1e6:.2f}M")
    return model


# Loss Functions
class MulticlassDiceLoss(nn.Module):
    """
    Soft Dice averaged over foreground classes only (class 0 ignored).
    Prevents trivially high scores on the large background area.
    """
    def __init__(self, num_classes=3, smooth=1.0):
        super().__init__()
        self.num_classes = num_classes
        self.smooth      = smooth

    def forward(self, logits, targets):
        probs      = F.softmax(logits, dim=1)
        targets_oh = F.one_hot(targets, self.num_classes)       # (B, H, W, C)
        targets_oh = targets_oh.permute(0, 3, 1, 2).float()     # (B, C, H, W)

        dice_losses = []
        for c in range(1, self.num_classes):                    # skip class 0
            p   = probs[:, c]
            t   = targets_oh[:, c]
            num = (p * t).sum()
            den = p.sum() + t.sum()
            dice_losses.append(1.0 - (2.0 * num + self.smooth) /
                                      (den + self.smooth))

        return torch.stack(dice_losses).mean()


class CombinedLoss(nn.Module):
    def __init__(self):
        super().__init__()
        weights   = torch.tensor(CFG.CE_CLASS_WEIGHTS, dtype=torch.float).to(DEVICE)
        self.ce   = nn.CrossEntropyLoss(weight=weights)
        self.dice = MulticlassDiceLoss(num_classes=CFG.NUM_CLASSES)

    def forward(self, logits, targets):
        return (CFG.CE_WEIGHT   * self.ce(logits, targets) +
                CFG.DICE_WEIGHT * self.dice(logits, targets))


# Metrics
@torch.no_grad()
def multiclass_dice(logits, targets, smooth=1e-6):
    """Returns mean fg dice (float) + per-class list (classes 1, 2)."""
    preds  = logits.argmax(dim=1)
    scores = []
    for c in range(1, CFG.NUM_CLASSES):
        p   = (preds == c).float()
        t   = (targets == c).float()
        num = (p * t).sum()
        den = p.sum() + t.sum()
        scores.append(((2.0 * num + smooth) / (den + smooth)).item())
    return float(np.mean(scores)), scores


@torch.no_grad()
def multiclass_iou(logits, targets, smooth=1e-6):
    """Returns mean fg IoU (float) + per-class list (classes 1, 2)."""
    preds  = logits.argmax(dim=1)
    scores = []
    for c in range(1, CFG.NUM_CLASSES):
        p     = (preds == c).float()
        t     = (targets == c).float()
        inter = (p * t).sum()
        union = p.sum() + t.sum() - inter
        scores.append(((inter + smooth) / (union + smooth)).item())
    return float(np.mean(scores)), scores


# Early Stopping
class EarlyStopping:
    """
    Monitors val_loss and triggers when improvement stalls.

    Args:
        patience   : epochs to wait after last improvement before stopping.
        min_delta  : minimum decrease in val_loss to count as an improvement.
        restore_best : if True, restores the best model weights on stop.
        verbose    : print status on each epoch.
    """

    def __init__(self, patience: int  = CFG.ES_PATIENCE,
                       min_delta: float = CFG.ES_MIN_DELTA,
                       restore_best: bool = CFG.ES_RESTORE_BEST,
                       verbose: bool = True):
        self.patience     = patience
        self.min_delta    = min_delta
        self.restore_best = restore_best
        self.verbose      = verbose

        self.best_loss    = float("inf")
        self.best_weights = None          # in-memory snapshot of best state_dict
        self.counter      = 0
        self.stopped_epoch = None

    def step(self, val_loss: float, model: nn.Module) -> bool:
        """
        Call once per epoch.
        Returns True  → training should stop.
        Returns False → training should continue.
        """
        improved = val_loss < (self.best_loss - self.min_delta)

        if improved:
            if self.verbose:
                print(f"  ✅ ES: val_loss improved "
                      f"{self.best_loss:.5f} → {val_loss:.5f}  "
                      f"(Δ={self.best_loss - val_loss:.5f})  counter reset.")
            self.best_loss    = val_loss
            self.best_weights = {k: v.cpu().clone()
                                 for k, v in model.state_dict().items()}
            self.counter      = 0
        else:
            self.counter += 1
            if self.verbose:
                print(f"  ⏳ ES: no improvement for {self.counter}/{self.patience} epochs  "
                      f"(best={self.best_loss:.5f}, current={val_loss:.5f})")

            if self.counter >= self.patience:
                self.stopped_epoch = True
                if self.restore_best and self.best_weights is not None:
                    model.load_state_dict(
                        {k: v.to(DEVICE) for k, v in self.best_weights.items()}
                    )
                    print(f"\n  🔁 ES: best weights restored  "
                          f"(best val_loss={self.best_loss:.5f})")
                print(f"\n  🛑 Early stopping triggered after "
                      f"{self.patience} epochs without improvement.\n")
                return True   # stop

        return False          # continue


# Train / Val Loops
def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    tot_loss = tot_dice = tot_iou = 0.0

    for images, masks in loader:
        images = images.to(DEVICE, non_blocking=True)
        masks  = masks.to(DEVICE,  non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=CFG.AMP):
            logits = model(images)
            loss   = criterion(logits, masks)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        tot_loss += loss.item()
        d, _      = multiclass_dice(logits, masks)
        iou, _    = multiclass_iou(logits, masks)
        tot_dice  += d
        tot_iou   += iou

    n = len(loader)
    return tot_loss / n, tot_dice / n, tot_iou / n


@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    tot_loss = tot_dice = tot_iou = 0.0
    per_class_dice = [0.0] * (CFG.NUM_CLASSES - 1)
    per_class_iou  = [0.0] * (CFG.NUM_CLASSES - 1)

    for images, masks in loader:
        images = images.to(DEVICE, non_blocking=True)
        masks  = masks.to(DEVICE,  non_blocking=True)

        with autocast(enabled=CFG.AMP):
            logits = model(images)
            loss   = criterion(logits, masks)

        tot_loss += loss.item()
        d, dc     = multiclass_dice(logits, masks)
        iou, ic   = multiclass_iou(logits, masks)
        tot_dice  += d
        tot_iou   += iou
        for i in range(len(dc)):
            per_class_dice[i] += dc[i]
            per_class_iou[i]  += ic[i]

    n   = len(loader)
    pcd = [v / n for v in per_class_dice]
    pci = [v / n for v in per_class_iou]
    return tot_loss / n, tot_dice / n, tot_iou / n, pcd, pci


# Checkpoint & Logger
def save_checkpoint(model, optimizer, epoch, val_loss, tag="best"):
    path = os.path.join(CFG.CKPT_DIR, f"convunext_3cls_{tag}.pth")
    torch.save({
        "epoch"      : epoch,
        "model_state": model.state_dict(),
        "optim_state": optimizer.state_dict(),
        "val_loss"   : val_loss,
        "backbone"   : CFG.BACKBONE,
        "num_classes": CFG.NUM_CLASSES,
        "img_size"   : CFG.IMG_SIZE,
    }, path)
    print(f"  💾 Checkpoint → {path}  (val_loss={val_loss:.4f})")


def load_checkpoint(model, optimizer=None, tag="best"):
    path = os.path.join(CFG.CKPT_DIR, f"convunext_3cls_{tag}.pth")
    ckpt = torch.load(path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    if optimizer:
        optimizer.load_state_dict(ckpt["optim_state"])
    print(f"  📥 Loaded '{path}'  "
          f"(epoch {ckpt['epoch']}, val_loss={ckpt['val_loss']:.4f})")
    return ckpt["epoch"]


class MetricsLogger:
    def __init__(self):
        self.h = {k: [] for k in [
            "train_loss", "val_loss",
            "train_dice", "val_dice",
            "train_iou",  "val_iou",
            "val_dice_c1", "val_dice_c2",
            "val_iou_c1",  "val_iou_c2",
        ]}

    def update(self, tm, vm, pcd, pci):
        self.h["train_loss"].append(tm[0])
        self.h["train_dice"].append(tm[1])
        self.h["train_iou"].append(tm[2])
        self.h["val_loss"].append(vm[0])
        self.h["val_dice"].append(vm[1])
        self.h["val_iou"].append(vm[2])
        self.h["val_dice_c1"].append(pcd[0] if len(pcd) > 0 else 0)
        self.h["val_dice_c2"].append(pcd[1] if len(pcd) > 1 else 0)
        self.h["val_iou_c1"].append(pci[0]  if len(pci) > 0 else 0)
        self.h["val_iou_c2"].append(pci[1]  if len(pci) > 1 else 0)

    def save(self):
        np.save(os.path.join(CFG.LOG_DIR, "metrics_3cls.npy"), self.h)

    def plot(self, stopped_at=None):
        epochs = range(1, len(self.h["train_loss"]) + 1)
        fig, axes = plt.subplots(2, 3, figsize=(18, 9))

        for ax, (tr_k, vl_k, title) in zip(axes[0], [
            ("train_loss", "val_loss", "Loss"),
            ("train_dice", "val_dice", "Mean Dice (fg classes)"),
            ("train_iou",  "val_iou",  "Mean IoU  (fg classes)"),
        ]):
            ax.plot(epochs, self.h[tr_k], label="Train", lw=2)
            ax.plot(epochs, self.h[vl_k], label="Val",   lw=2, ls="--")
            if stopped_at:
                ax.axvline(stopped_at, color="red", ls=":", lw=1.5,
                           label=f"Early stop (ep {stopped_at})")
            ax.set_title(title, fontweight="bold")
            ax.set_xlabel("Epoch")
            ax.legend(); ax.grid(alpha=0.3)

        for ax, (k_dice, k_iou, cls_name, col) in zip(axes[1], [
            ("val_dice_c1", "val_iou_c1", "Class 1", "tab:blue"),
            ("val_dice_c2", "val_iou_c2", "Class 2", "tab:orange"),
        ]):
            ax.plot(epochs, self.h[k_dice], label="Dice", lw=2, color=col)
            ax.plot(epochs, self.h[k_iou],  label="IoU",  lw=2, color=col,
                    ls="--")
            if stopped_at:
                ax.axvline(stopped_at, color="red", ls=":", lw=1.5)
            ax.set_title(f"{cls_name} — Dice & IoU", fontweight="bold")
            ax.set_xlabel("Epoch")
            ax.legend(); ax.grid(alpha=0.3)

        axes[1][2].axis("off")

        plt.suptitle("ConvUNeXt 3-Class Training Curves",
                     fontsize=15, fontweight="bold")
        plt.tight_layout()
        out = os.path.join(CFG.LOG_DIR, "training_curves_3cls.png")
        plt.savefig(out, dpi=150)
        plt.show()
        print(f"  📊 Saved {out}")


# Main Training Loop
def train():
    print("\n" + "=" * 60)
    print("  ConvUNeXt 3-Class — Training")
    print("=" * 60)

    train_loader, val_loader = build_loaders()
    model     = build_model()
    criterion = CombinedLoss()

    # Differential LR: encoder fine-tuned at 10× lower rate than decoder
    enc_params = list(model.encoder.parameters())
    dec_params = [p for p in model.parameters()
                  if not any(p is e for e in enc_params)]

    optimizer = torch.optim.AdamW([
        {"params": enc_params, "lr": CFG.LR * 0.1},
        {"params": dec_params, "lr": CFG.LR},
    ], weight_decay=CFG.WEIGHT_DECAY)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=CFG.NUM_EPOCHS, eta_min=CFG.LR_MIN
    )
    scaler  = GradScaler(enabled=CFG.AMP)
    logger  = MetricsLogger()
    es      = EarlyStopping(
                  patience     = CFG.ES_PATIENCE,
                  min_delta    = CFG.ES_MIN_DELTA,
                  restore_best = CFG.ES_RESTORE_BEST,
              )

    best_val_loss  = float("inf")
    stopped_at_ep  = None

    for epoch in range(1, CFG.NUM_EPOCHS + 1):
        t0 = time.time()

        tm                         = train_one_epoch(
            model, train_loader, optimizer, criterion, scaler
        )
        vl, vd, vi, pcd, pci      = validate(model, val_loader, criterion)
        scheduler.step()

        logger.update(tm, (vl, vd, vi), pcd, pci)

        tl, td, ti = tm
        elapsed    = time.time() - t0
        lr_now     = optimizer.param_groups[1]["lr"]
        pc_str     = " | ".join(f"C{i+1}:{pcd[i]:.3f}"
                                 for i in range(len(pcd)))

        print(
            f"\nEp {epoch:03d}/{CFG.NUM_EPOCHS} | "
            f"T-Loss {tl:.4f}  T-Dice {td:.4f}  T-IoU {ti:.4f} | "
            f"V-Loss {vl:.4f}  V-Dice {vd:.4f}  ({pc_str}) | "
            f"LR {lr_now:.2e}  ⏱ {elapsed:.1f}s"
        )

        # ── Checkpoint best model ─────────────────────────────────────────
        if CFG.SAVE_BEST and vl < best_val_loss:
            best_val_loss = vl
            save_checkpoint(model, optimizer, epoch, vl, tag="best")

        if epoch % 10 == 0:
            #save_checkpoint(model, optimizer, epoch, vl, tag=f"ep{epoch:03d}")
            save_checkpoint(model, optimizer, epoch, vl, tag=f"ep")

        # ── Early stopping check (called after checkpoint so best is saved) ─
        if es.step(vl, model):
            stopped_at_ep = epoch
            # After es.step() restores best weights into model,
            # write them to disk so "best" is always consistent with
            # what the model actually holds at the end of training.
            print(f"  💾 Saving final best weights after restoration …")
            save_checkpoint(model, optimizer, epoch, es.best_loss, tag="best")
            break

    logger.save()
    logger.plot(stopped_at=stopped_at_ep)

    if stopped_at_ep:
        print(f"✅ Training ended early at epoch {stopped_at_ep}.")
    else:
        print(f"✅ Training complete — all {CFG.NUM_EPOCHS} epochs finished.")


# Inference & Visualisation
CLASS_COLORS = np.array([
    [0,   0,   0,   0  ],    # 0 = background  : transparent
    [0,   120, 255, 180],    # 1 = class 1     : blue
    [255, 60,  60,  180],    # 2 = class 2     : red
], dtype=np.uint8)


def _colorise_mask(pred_mask: np.ndarray) -> np.ndarray:
    return CLASS_COLORS[pred_mask]


@torch.no_grad()
def predict_single(model, image_path, return_probs=False):
    """
    Run inference on a single image path.
    Alpha channel is stripped automatically by load_image_rgb().
    """
    original = load_image_rgb(image_path)   # always (H, W, 3)
    H, W     = original.shape[:2]

    aug = val_transform(image=original,
                        mask=np.zeros((H, W), dtype=np.int64))
    x   = aug["image"].unsqueeze(0).float().to(DEVICE)

    model.eval()
    with autocast(enabled=CFG.AMP):
        logits = model(x)

    probs = F.softmax(logits, dim=1).squeeze(0).cpu().numpy()   # (C, H, W)
    pred  = probs.argmax(axis=0).astype(np.float32)
    pred  = cv2.resize(pred, (W, H),
                       interpolation=cv2.INTER_NEAREST).astype(np.int64)

    if return_probs:
        return original, pred, probs
    return original, pred


def visualise_predictions(model, n_samples=4, show_class3=True):
    def _gather(folder, n):
        return sorted([
            str(p) for p in Path(folder).iterdir()
            if p.suffix.lower() in SUPPORTED_EXTS
        ])[:n]

    c1_imgs  = _gather(CFG.CLASS1_IMG_DIR,  n_samples)
    c1_masks = _gather(CFG.CLASS1_MASK_DIR, n_samples)
    c2_imgs  = _gather(CFG.CLASS2_IMG_DIR,  n_samples)
    c2_masks = _gather(CFG.CLASS2_MASK_DIR, n_samples)
    c3_imgs  = _gather(CFG.CLASS3_IMG_DIR,  n_samples) if show_class3 else []

    rows  = len(c1_imgs) + len(c2_imgs) + len(c3_imgs)
    fig, axes = plt.subplots(rows, 3, figsize=(15, rows * 4))
    if rows == 1:
        axes = [axes]

    row = 0

    def _plot_row(img_path, mask_path=None, label=""):
        orig, pred = predict_single(model, img_path)
        overlay    = _colorise_mask(pred)

        axes[row][0].imshow(orig)
        axes[row][0].set_title(f"{label} — Image", fontsize=10)
        axes[row][0].axis("off")

        if mask_path and Path(mask_path).exists():
            # Use load_mask_binary so RGBA masks are also handled correctly
            gt_label = 1 if "class_1" in str(mask_path) else 2
            gt = load_mask_binary(mask_path, fg_label=gt_label)
            # Display: bg=0 (black), fg=255 (white)
            gt_vis = (gt > 0).astype(np.uint8) * 255
            axes[row][1].imshow(gt_vis, cmap="gray")
            axes[row][1].set_title("Ground Truth", fontsize=10)
        else:
            axes[row][1].imshow(
                np.zeros(orig.shape[:2], dtype=np.uint8), cmap="gray"
            )
            axes[row][1].set_title("GT (none)", fontsize=10, color="gray")
        axes[row][1].axis("off")

        axes[row][2].imshow(orig)
        axes[row][2].imshow(overlay, alpha=0.55)
        axes[row][2].set_title("Prediction overlay", fontsize=10)
        axes[row][2].axis("off")

    for ip, mp in zip(c1_imgs, c1_masks):
        _plot_row(ip, mp, "Class 1"); row += 1
    for ip, mp in zip(c2_imgs, c2_masks):
        _plot_row(ip, mp, "Class 2"); row += 1
    for ip in c3_imgs:
        _plot_row(ip, None, "Class 3 (bg)"); row += 1

    from matplotlib.patches import Patch
    legend = [
        Patch(facecolor=(0, 120/255, 1, 0.7),      label="Class 1"),
        Patch(facecolor=(1, 60/255, 60/255, 0.7),  label="Class 2"),
    ]
    fig.legend(handles=legend, loc="lower center", ncol=2,
               fontsize=11, framealpha=0.9)

    plt.suptitle("ConvUNeXt 3-Class Predictions",
                 fontsize=14, fontweight="bold")
    plt.tight_layout(rect=[0, 0.04, 1, 1])
    out = os.path.join(CFG.LOG_DIR, "predictions_3cls.png")
    plt.savefig(out, dpi=150)
    plt.show()
    print(f"  🖼  Saved {out}")


def visualise_confidence(model, image_path):
    """Per-class softmax probability maps for a single image."""
    original, pred, probs = predict_single(model, image_path,
                                           return_probs=True)
    class_names = ["Background", "Class 1", "Class 2"]
    fig, axes   = plt.subplots(1, CFG.NUM_CLASSES + 1,
                                figsize=(5 * (CFG.NUM_CLASSES + 1), 5))

    axes[0].imshow(original)
    axes[0].set_title("Input", fontweight="bold"); axes[0].axis("off")

    for c in range(CFG.NUM_CLASSES):
        im = axes[c + 1].imshow(probs[c], cmap="hot", vmin=0, vmax=1)
        axes[c + 1].set_title(f"P({class_names[c]})", fontweight="bold")
        axes[c + 1].axis("off")
        plt.colorbar(im, ax=axes[c + 1], fraction=0.046, pad=0.04)

    plt.suptitle("Softmax Confidence Maps", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()


# Dry Run (Shape Check)
def dry_run():
    print("=" * 60)
    print("  ConvUNeXt 3-Class — Dry Run")
    print("=" * 60)

    model = build_model()
    model.eval()

    dummy_img  = torch.randn(2, 3, CFG.IMG_SIZE, CFG.IMG_SIZE).to(DEVICE)
    dummy_mask = torch.randint(0, CFG.NUM_CLASSES,
                               (2, CFG.IMG_SIZE, CFG.IMG_SIZE)).to(DEVICE)

    with autocast(enabled=CFG.AMP):
        out = model(dummy_img)

    crit = CombinedLoss()
    loss = crit(out, dummy_mask)

    d, per_c = multiclass_dice(out, dummy_mask)

    print(f"  ✅ Input  : {tuple(dummy_img.shape)}")
    print(f"  ✅ Output : {tuple(out.shape)}")    # (2, 3, 512, 512)
    print(f"  ✅ Loss   : {loss.item():.4f}")
    print(f"  ✅ Mean Dice : {d:.4f}  |  "
          f"Per-class : {[f'{v:.4f}' for v in per_c]}")

    # Verify RGBA stripping on a dummy in-memory image
    dummy_rgba = np.random.randint(0, 255, (64, 64, 4), dtype=np.uint8)
    dummy_path = "/tmp/_test_rgba.png"
    cv2.imwrite(dummy_path, dummy_rgba)
    rgb_out = load_image_rgb(dummy_path)
    assert rgb_out.shape == (64, 64, 3), "RGBA → RGB conversion failed!"
    print(f"  ✅ RGBA strip : (64,64,4) → {rgb_out.shape}  ✓")

    print("  🎉 Dry run passed!\n")


# Entry Points
# 1. Shape + RGBA check
dry_run()

# 2. Training (early stopping embedded)
train()

# 3. Visualise after training
model = build_model()
load_checkpoint(model, tag="best")

visualise_predictions(model, n_samples=4, show_class3=True)


# OPTIONAL: per-class confidence heatmaps
# visualise_confidence(model, "class_1/your_image.png")

Output hidden; open in https://colab.research.google.com to view.